In [ ]:
# ============================================================
# Configuration
# ============================================================

from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# -------------------------
# Dataset configuration
# -------------------------

DATASET_SIZE = 25000

DATA_DIR = PROJECT_ROOT / "data"
DATASET_PATH = DATA_DIR / f"gw_dataset_{DATASET_SIZE}.npz"

FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

# -------------------------
# Model configuration
# -------------------------

MODEL_DIR = PROJECT_ROOT / "models" / f"bayesflow_model_{DATASET_SIZE}"

# -------------------------
# Training configuration
# -------------------------

MAX_SAMPLES = None        # Use the entire dataset
EPOCHS = 25              # Increase later if needed
BATCH_SIZE = 64

print("Project Root :", PROJECT_ROOT)
print("Dataset      :", DATASET_PATH)
print("Model Dir    :", MODEL_DIR)
print("Epochs       :", EPOCHS)
print("Batch Size   :", BATCH_SIZE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.model import (
    PARAMETER_NAMES,
    history_to_losses,
    load_npz_dataset,
    sample_posterior,
    train_workflow,
)

# Notebook 02 saves the signals and parameters separately.
# Combine them into one file before training.
if not DATASET_PATH.exists():
    waveforms_path = DATA_DIR / f"waveforms_{DATASET_SIZE}.npy"
    parameters_path = DATA_DIR / f"parameters_{DATASET_SIZE}.npy"

    if not waveforms_path.exists() or not parameters_path.exists():
        raise FileNotFoundError(
            f"Could not find {DATASET_PATH.name}, {waveforms_path.name}, or {parameters_path.name}. "
            "Run notebooks/02_generate_dataset.ipynb first."
        )

    X = np.load(waveforms_path, mmap_mode="r")
    theta = np.load(parameters_path, mmap_mode="r")
    expected_x_shape = (DATASET_SIZE, 8192)
    expected_theta_shape = (DATASET_SIZE, 6)

    if X.shape != expected_x_shape or theta.shape != expected_theta_shape:
        raise ValueError(
            f"Unexpected notebook 02 output shapes: X={X.shape}, theta={theta.shape}; "
            f"expected {expected_x_shape} and {expected_theta_shape}."
        )
    if not np.isfinite(X).all() or not np.isfinite(theta).all():
        raise ValueError("Generated dataset contains NaN or infinite values.")
    if not np.all(theta[:, 0] >= theta[:, 1]):
        raise ValueError("Generated dataset violates the required ordering m1 >= m2.")

    np.savez(DATASET_PATH, X=X, theta=theta)
    print("Created training dataset:", DATASET_PATH)
    del X, theta

dataset = load_npz_dataset(DATASET_PATH)
print("strain:", dataset["strain"].shape)
print("parameters:", dataset["parameters"].shape)
print("parameter order:", PARAMETER_NAMES)

In [ ]:
MAX_SAMPLES = None
EPOCHS = 25
BATCH_SIZE = 64

workflow, history, train_data, val_data, test_data = train_workflow(
    dataset_path=DATASET_PATH,
    model_dir=MODEL_DIR,
    max_samples=MAX_SAMPLES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

print("Train      :", train_data["strain"].shape)
print("Validation :", val_data["strain"].shape)
print("Test       :", test_data["strain"].shape)

In [ ]:
# ----------------------------------------------------
# Posterior distribution on ONE UNSEEN TEST SAMPLE
# ----------------------------------------------------

test_strain = test_data["strain"][0, :, 0]
true_theta = test_data["parameters"][0]

posterior_samples = sample_posterior(
    workflow,
    test_strain,
    num_samples=1000,
)

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.ravel()

for i, name in enumerate(PARAMETER_NAMES):
    axes[i].hist(
        posterior_samples[:, i],
        bins=35,
        density=True,
        alpha=0.75,
    )

    axes[i].axvline(
        true_theta[i],
        color="black",
        linestyle="--",
        linewidth=2,
        label="True value",
    )

    axes[i].set_title(name)
    axes[i].grid(alpha=0.2)

axes[0].legend()

fig.suptitle("Posterior samples for one UNSEEN test signal")

fig.tight_layout()

posterior_path = FIGURE_DIR / "posterior_example_test.png"
fig.savefig(posterior_path, dpi=160)

posterior_path

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import clear_output

# -------------------------------------------------
# Evaluate on ALL unseen test samples (2500)
# -------------------------------------------------

n_examples = len(test_data["strain"])   # = 2500

true_params = []
pred_params = []

for i in tqdm(range(n_examples), desc="Evaluating test set"):

    strain = test_data["strain"][i, :, 0]
    true_theta = test_data["parameters"][i]

    posterior = sample_posterior(
        workflow,
        strain,
        num_samples=500,
    )

    pred_theta = posterior.mean(axis=0)

    true_params.append(true_theta)
    pred_params.append(pred_theta)

    # remove BayesFlow sampling messages every iteration
    clear_output(wait=True)

true_params = np.array(true_params)
pred_params = np.array(pred_params)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.ravel()

for i, name in enumerate(PARAMETER_NAMES):

    axes[i].scatter(
        true_params[:, i],
        pred_params[:, i],
        s=8,
        alpha=0.45,
    )

    mn = min(true_params[:, i].min(), pred_params[:, i].min())
    mx = max(true_params[:, i].max(), pred_params[:, i].max())

    axes[i].plot([mn, mx], [mn, mx], "r--", linewidth=2)

    axes[i].set_xlabel("True")
    axes[i].set_ylabel("Predicted")
    axes[i].set_title(name)
    axes[i].grid(alpha=0.3)

fig.suptitle("Predicted vs True Parameters (2500 Unseen Test Samples)")
fig.tight_layout()

scatter_path = FIGURE_DIR / "predicted_vs_true_2500.png"
fig.savefig(scatter_path, dpi=160)

print("Saved to:", scatter_path)

plt.show()

In [ ]:
from src.model import load_workflow

workflow = load_workflow(MODEL_DIR)

print("Model loaded successfully!")

In [ ]:
from src.model import load_npz_dataset, split_dataset

dataset = load_npz_dataset(DATASET_PATH)

train_data, val_data, test_data = split_dataset(
    dataset,
    validation_fraction=0.1,
    test_fraction=0.1,
    seed=2026
)

print("Train      :", train_data["strain"].shape)
print("Validation :", val_data["strain"].shape)
print("Test       :", test_data["strain"].shape)

In [ ]:
true_params = np.zeros((1000, 6), dtype=np.float32)
pred_params = np.zeros((1000, 6), dtype=np.float32)

for i in tqdm(range(1000)):
    strain = test_data["strain"][i, :, 0]

    posterior = sample_posterior(
        workflow,
        strain,
        num_samples=500,
    )

    true_params[i] = test_data["parameters"][i]
    pred_params[i] = posterior.mean(axis=0)

In [ ]:
print(true_params.shape)
print(pred_params.shape)

In [ ]:
import bayesflow as bf

fig = bf.diagnostics.calibration_histogram(
    estimates=pred_params,
    targets=true_params,
    variable_names=list(PARAMETER_NAMES)
)

fig.savefig(FIGURE_DIR / "calibration_histogram.png", dpi=160)

plt.show()

In [ ]:
fig = bf.diagnostics.calibration_ecdf(
    estimates=pred_params,
    targets=true_params,
    variable_names=list(PARAMETER_NAMES)
)

fig.savefig(FIGURE_DIR / "calibration_ecdf.png", dpi=160)

plt.show()